# 00 — Geração de Mensagens Sintéticas**TCC: Detecção de Smishing em Idosos com Modelos de Linguagem Natural**> **Notebook de uso único.** Roda uma vez, produz um CSV versionado, e não faz> parte do fluxo recorrente. Isso mantém o `01_dados` em CPU e evita disputar a> cota de GPU com os notebooks 03 e 04.**Respaldo na metodologia:** a seção 4.4.2 prevê "mensagens fraudulentas reais(quando possível) **ou simuladas com base em evidências**", com rotulagem manual.Este notebook implementa a parte de geração; a **validação manual é obrigatória**e acontece na seção 5, antes de qualquer mensagem entrar no corpus.### Regra que não pode ser quebradaCada mensagem gerada carrega o `id_semente` da mensagem real que a originou.É esse campo que impede, no notebook 01, que variações da mesma semente caiamem lados opostos do split — o vazamento mais provável e mais destrutivo desteprojeto.### Saída`data/synthetic/sinteticas_validadas.csv` — apenas as mensagens aprovadas narevisão manual.

## 1. Setup

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
# No Colab CADA notebook roda em um runtime próprio: instalar dependências
# em um notebook não vale para os outros. Por isso esta célula se repete em
# todos, e não existe um "notebook de instalação".

REPO = 'https://github.com/FelypeSR/TCC_Cristian.git'   # ← ajuste aqui

!git clone -q {REPO} /content/TCC_Cristian 2>/dev/null || (cd /content/TCC_Cristian && git pull -q)
!pip install -q -r /content/TCC_Cristian/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/TCC_Cristian/src')

import config as CFG
CFG.fixar_seeds()
CFG.criar_pastas()
CFG.resumo()

## 2. Carregamento das sementesAs sementes são as mensagens **reais** curadas a partir de Bortot et al. (2024)e das cartilhas do CERT.br / GOV.BR. Precisam já estar padronizadas em`data/raw/` com as colunas `texto`, `rotulo` e `tipo_golpe`.

In [ ]:
import pandas as pd
import os

CAMINHO_SEMENTES = f"{CFG.PATHS['raw']}/sementes_pt.csv"

if not os.path.isfile(CAMINHO_SEMENTES):
    raise FileNotFoundError(
        f"Sementes não encontradas em {CAMINHO_SEMENTES}.\n"
        "Faça o upload do CSV curado com as colunas: texto, rotulo, tipo_golpe"
    )

sementes = pd.read_csv(CAMINHO_SEMENTES, encoding='utf-8')

# Só mensagens de golpe são usadas como semente. Gerar mensagens "legítimas"
# sintéticas é arriscado: o modelo tende a produzir textos genéricos e
# artificiais, e o classificador aprenderia a separar estilo, não conteúdo.
sementes = sementes[sementes[CFG.COL_ROTULO] == CFG.CLASSE_POSITIVA].reset_index(drop=True)

# O id da semente vem do CONTEÚDO da mensagem, não da posição na planilha.
# É esse id que o notebook 01 usa para saber de qual mensagem real cada
# variação nasceu. Um id posicional não sobrevive à travessia entre os dois
# notebooks — eles leem arquivos diferentes — e o filtro anti-vazamento
# acabaria descartando a augmentation inteira em silêncio.
import preprocessing as pp
sementes['id_semente'] = sementes[CFG.COL_TEXTO].apply(pp.id_mensagem)

print(f'Sementes de smishing: {len(sementes)}')
print(sementes['tipo_golpe'].value_counts().to_string())
sementes.head(3)

## 3. Carregamento do Llama quantizado

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Token nos Secrets do Colab (ícone da chave na barra lateral), nunca no código
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',          # NF4 é otimizado para pesos de redes neurais
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

MODEL_ID = CFG.MODELOS['llama']
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
modelo = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', token=HF_TOKEN,
)
modelo.eval()

print(f'VRAM em uso: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## 4. GeraçãoPara cada semente, o modelo gera variações **do mesmo tipo de golpe**, mudandoa redação mas preservando o mecanismo de persuasão. Temperatura alta aqui éproposital: queremos diversidade, não a resposta mais provável.

In [ ]:
from tqdm.auto import tqdm

N_VARIACOES = 4      # por semente
TEMPERATURA = 0.9    # alta de propósito — o objetivo é variedade

INSTRUCAO = (
    'Você ajuda a construir um conjunto de dados acadêmico para TREINAR um '
    'detector de golpes por SMS que protege pessoas idosas no Brasil.\n\n'
    'A partir da mensagem de golpe abaixo, escreva {n} variações diferentes '
    'do MESMO tipo de golpe ({tipo}). Mude as palavras, a instituição citada '
    'e os valores, mas mantenha o mecanismo de persuasão.\n\n'
    'Regras:\n'
    '- Cada variação em uma linha, numerada de 1 a {n}\n'
    '- Máximo de 200 caracteres por mensagem\n'
    '- Português do Brasil, no estilo real de SMS\n'
    '- Não use dados de pessoas reais: sem nomes próprios, CPF ou contas verdadeiras\n'
    '- Não escreva nada além das mensagens numeradas\n\n'
    'Mensagem original: {texto}'
)


def gerar_variacoes(texto, tipo, n=N_VARIACOES):
    mensagens = [{'role': 'user', 'content': INSTRUCAO.format(n=n, tipo=tipo, texto=texto)}]
    entrada = tok.apply_chat_template(
        mensagens, tokenize=True, add_generation_prompt=True, return_tensors='pt',
    ).to(modelo.device)

    with torch.no_grad():
        saida = modelo.generate(
            entrada, max_new_tokens=400, do_sample=True,
            temperature=TEMPERATURA, top_p=0.95,
            pad_token_id=tok.eos_token_id,
        )

    bruto = tok.decode(saida[0][entrada.shape[-1]:], skip_special_tokens=True)

    # Extrai as linhas numeradas, descartando o resto
    linhas = []
    for linha in bruto.split('\n'):
        linha = linha.strip()
        if linha and linha[0].isdigit():
            texto_limpo = linha.lstrip('0123456789.)-  ').strip()
            if len(texto_limpo) > 20:
                linhas.append(texto_limpo)
    return linhas[:n]


geradas = []
for _, s in tqdm(sementes.iterrows(), total=len(sementes), desc='Gerando'):
    for variacao in gerar_variacoes(s[CFG.COL_TEXTO], s['tipo_golpe']):
        geradas.append({
            'texto': variacao,
            'rotulo': CFG.CLASSE_POSITIVA,
            'tipo_golpe': s['tipo_golpe'],
            'id_semente': s['id_semente'],   # ← rastreabilidade obrigatória
            'fonte': 'sintetica',
            'idioma': 'pt',
        })

df_geradas = pd.DataFrame(geradas)
print(f'\nGeradas: {len(df_geradas)} mensagens a partir de {len(sementes)} sementes')

## 5. Deduplicação e preparação para a revisão manualO modelo repete formulações com frequência. Deduplicamos antes de gastar tempode revisão humana com mensagens idênticas.

In [ ]:
import preprocessing as pp

# Deduplica entre as geradas e contra as próprias sementes
df_geradas['_chave'] = df_geradas['texto'].apply(pp.chave_dedup)
chaves_sementes = set(sementes[CFG.COL_TEXTO].apply(pp.chave_dedup))

antes = len(df_geradas)
df_geradas = df_geradas.drop_duplicates(subset='_chave')
df_geradas = df_geradas[~df_geradas['_chave'].isin(chaves_sementes)]
df_geradas = df_geradas.drop(columns='_chave').reset_index(drop=True)

print(f'Duplicatas removidas: {antes - len(df_geradas)}')
print(f'Restantes para revisão: {len(df_geradas)}')

df_geradas['id'] = [f'sint_{i}' for i in range(len(df_geradas))]
df_geradas['aprovada'] = ''      # ← preencher na revisão manual: 1 = aprovada, 0 = descartada

CAMINHO_REVISAO = f"{CFG.PATHS['synthetic']}/sinteticas_para_revisao.csv"
df_geradas.to_csv(CAMINHO_REVISAO, index=False, encoding='utf-8')
print(f'\nArquivo de revisão: {CAMINHO_REVISAO}')

## 6. ⚠️ Revisão manual — etapa obrigatóriaA seção 4.4.2 da monografia exige rotulagem manual. Além disso, mensagensgeradas por LLM frequentemente saem implausíveis, fora do português brasileirocoloquial, ou desalinhadas do tipo de golpe pedido.**Faça agora, fora do notebook:**1. Abra `data/synthetic/sinteticas_para_revisao.csv` no Planilhas Google2. Para cada linha, preencha a coluna `aprovada`:   - `1` — plausível como SMS real de golpe, coerente com o `tipo_golpe`   - `0` — implausível, truncada, genérica demais ou fora do tipo3. Corrija o `tipo_golpe` quando o modelo tiver desviado4. Salve o arquivo **com o mesmo nome**Registre no texto da monografia quantas foram geradas, quantas aprovadas e qualo critério de descarte — a taxa de aprovação é resultado, não detalhe operacional.Só depois disso execute a célula abaixo.

In [ ]:
revisadas = pd.read_csv(CAMINHO_REVISAO, encoding='utf-8')

if revisadas['aprovada'].isna().all() or (revisadas['aprovada'].astype(str).str.strip() == '').all():
    raise ValueError(
        'A coluna "aprovada" está vazia — a revisão manual ainda não foi feita.\n'
        'Ver instruções na célula acima.'
    )

aprovadas = revisadas[revisadas['aprovada'].astype(str).str.strip() == '1'].copy()
aprovadas = aprovadas.drop(columns=['aprovada'])

taxa = len(aprovadas) / len(revisadas) if len(revisadas) else 0
print(f'Geradas   : {len(revisadas)}')
print(f'Aprovadas : {len(aprovadas)}  ({taxa:.1%})')
print(f'Descartadas: {len(revisadas) - len(aprovadas)}')
print('\nPor tipo de golpe:')
print(aprovadas['tipo_golpe'].value_counts().to_string())

aprovadas.to_csv(CFG.SINTETICAS, index=False, encoding='utf-8')
print(f'\nSalvo em: {CFG.SINTETICAS}')
print('\nProssiga para o notebook 01_dados.ipynb.')